# C03_E02 — Ajuste FOPDT por curve_fit

**Capítulo:** 3 — Modelado básico de procesos para control  
**Identificador:** `C03_E02`  
**Objetivo pedagógico:** Identificar K, τ, θ desde una respuesta a escalón con ruido.  
**Librerías:** numpy, scipy.optimize, matplotlib

## Ejemplo industrial

Curva de reacción de un horno tras escalón en consigna. Identificación de FOPDT representativo.

---

> **Aviso de uso responsable.** Este notebook es didáctico. No es código de producción. Cualquier implementación real requiere validación independiente. Ver `docs/politica_uso_responsable.md`.

## 1. Definición FOPDT y datos sintéticos con ruido

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import os
np.random.seed(0)

def fopdt(t, K, tau, theta):
    y = np.zeros_like(t, dtype=float)
    mask = t > theta
    y[mask] = K*(1.0 - np.exp(-(t[mask] - theta)/tau))
    return y

# Datos sintéticos: K=2.0, tau=15.0, theta=5.0 con ruido
t = np.linspace(0, 100, 200)
y_real = fopdt(t, K=2.0, tau=15.0, theta=5.0) + 0.05*np.random.randn(t.size)

## 2. Ajuste de parámetros

In [2]:
popt, pcov = curve_fit(fopdt, t, y_real, p0=[1.0, 10.0, 1.0])
K_h, tau_h, theta_h = popt
print(f"K     = {K_h:.4f}  (real 2.0)")
print(f"tau   = {tau_h:.4f}  (real 15.0)")
print(f"theta = {theta_h:.4f}  (real 5.0)")
y_fit = fopdt(t, *popt)

K     = 2.0080  (real 2.0)
tau   = 15.4321  (real 15.0)
theta = 4.8093  (real 5.0)


## 3. Visualización del ajuste

In [3]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t, y_real, '.', label="Datos con ruido")
ax.plot(t, y_fit, '-', label=f"FOPDT ajustado: K={K_h:.2f}, τ={tau_h:.2f}, θ={theta_h:.2f}")
ax.set_xlabel("Tiempo (s)"); ax.set_ylabel("Salida")
ax.legend(); ax.grid(alpha=0.3)
ax.set_title("Ajuste FOPDT por curve_fit (Capítulo 3)")
out = '../../figures/cap03/fig_C03_F02_ajuste_fopdt.png'
os.makedirs(os.path.dirname(out), exist_ok=True)
fig.tight_layout(); fig.savefig(out, dpi=120)
plt.show()

## 4. Interpretación

El ajuste recupera K, τ y θ con error compatible con la dispersión introducida por el ruido. Un FOPDT identificado con esa precisión es suficiente para sintonizar un PID con SIMC, IMC o Lambda (Capítulo 10). **Atención:** una identificación con ruido sin filtrar produce parámetros poco fiables; en datos reales, filtrar ligeramente la curva antes del ajuste o usar identificación robusta.